# Motolinia：从阅读报告到复现审计（Python）

Lucia Motolinia, *Electoral Accountability and Particularistic Legislation: Evidence from an Electoral Reform in Mexico*, **American Political Science Review 115(1), 97–113**。

本 notebook 按“**原文主张 → 来源与设定 → 基准复现 → H2 主审计 → 短期组补充比较 → 影响范围与方法问答**”推进。全部计算重新读取`data/raw/` 中的官方文件，不依赖旧补充分析目录，也不把旧图片当作计算输入。

## 审计路线

| 顺序 | 要回答的问题 | 证据 |
|---|---|---|
| §1 | 原文究竟主张什么？质疑针对哪一项？ | 印刷页码、表号、假设及检查范围 |
| §2 | 数据和代码是否来自作者？ | 原文件、版本、哈希、处理编码、CTM 汇总 |
| §3 | Python 是否重建了原设定？ | 原始命令对照、H1/H3 复现和处理前检查 |
| §4 | H2 末年处理项是否能独立识别？ | 样本覆盖、标准 DID 对比、恒等式、矩阵秩、相同拟合、额外约束 |
| §5 | 短期组的直接比较说明什么？ | DID、少聚类推断、对照剔除、33 切点联合检验 |
| §6–7 | 哪些结论受影响？如何回填阅读报告？ | 影响评估、方法问答、措辞修订、待核实事项 |

**本次判断。**原文 H1/H3 的点估计可复现；H2 末年子样本中的处理项被州和年月固定效应完全吸收，不能在完整规格下单独识别。短期组补充比较的点估计为负，但少聚类推断与处理前变化限制其因果解释。两类问题分别论证。

**运行。**在仓库根目录安装 `requirements.txt`，运行 `python data/download.py` 下载并校验官方文件，再打开本 notebook 选择对应 Python 内核。公开 notebook 清空保存输出；本地运行会生成表和图，便于核验。

**复现边界。**沿用作者已经估计的主题份额，不重新训练 CTM。Python 执行记录不是原生 Stata 执行记录；作者当时实际省略哪一列、为何采用这些筛选，需要原始日志或进一步说明。数学上的线性依赖可以由公开数据与模型直接核验。

## 1. 原文主张、精确位置与审计对象

阅读报告先介绍研究问题：在政党仍掌握提名和竞选资源的情况下，连续连任机会是否使议员更关注可向特定地区或群体提供利益的议题？因变量是**会议中定向立法讨论的主题份额**，不直接等于法案通过数量、实际支出或政策质量。

原文提出三个预测：H1 关注长期激励相对无连任激励的总体变化；H2 关注这种变化是否在临近选举时更大；H3 关注长期相对短期激励的差异。H2 是关于选举周期的动态预测，不是对整条中介机制的完整识别。

下列为**期刊印刷页码**，不是 PDF 查看器页序。来源：[原文 DOI](https://doi.org/10.1017/S0003055420000672)。

| 原文位置 | 作者比较或论断 | 本 notebook 检查 |
|---|---|---|
| p.108，Table 2 | 长期激励 vs 无连任激励，H1 | 相同处理编码、固定效应、样本与点估计 |
| p.109，Figure 5 | H1 的处理前切点检查 | 重建 33 次独立安慰剂回归；不把“不显著”当作证明 |
| p.110，Figure 6 | 不同选举周期阶段的估计，H2 | 提取三个筛选子样本，重点审计末年处理项 |
| p.111，Table 3 | 长期激励 vs 短期激励，H3 | 相同比较、固定效应与点估计 |
| p.110，Figure 7 | H3 的处理前切点检查 | 重建作者的相应切点 |
| 我们的补充比较 | 短期激励 vs 无连任激励 | 直接重估，不用两个不同样本的系数相减代替 |

正文 Table 2 和 Table 3 未给出最后一项直接比较。这是补充分析的理由，不构成对作者选择性报告或主观动机的证明。

## 2. 原作者资料、数据与因变量

资料来源：[Harvard Dataverse V1](https://doi.org/10.7910/DVN/NOMC0H)；[论文 DOI](https://doi.org/10.1017/S0003055420000672)。

作者原始文件不随 Git 仓库分发。`data/download.py` 按 `data/sources.json` 直接从 Dataverse 获取到被忽略的 `data/raw/`，并检查 SHA256。来源条款及文件清单见 `data/README.md`。

| 原作者文件 | Dataverse 文件 ID | 用途 |
|---|---:|---|
| `allres.csv` | 3892153，下载参数 `format=original` | 完整会议级回归数据，含 450 个主题份额 |
| `replication_analysis.do` | 3892146 | 正文回归与图表 |
| `replication_appendix.do` | 3892135 | 作者附录分析 |
| `replication_cleaning_CTM.R` | 3892151 | 作者文本处理、主题模型、分组与结果汇总 |

下面核对本地文件哈希。三份代码下载时的 MD5 已与 Dataverse 元数据一致；CSV 的 SHA256 标识本次使用的原始 CSV 字节。后续若有意更换来源版本，应先更新并说明版本，而非绕过检查。

**日期记录。**初次下载完整 CSV 的日期没有作为本包的可靠元数据保存，不以整理日期替代。来源版本和具体文件用 DOI、文件 ID 及下方 SHA256 确定；本次运行时间在最后一节打印。

In [ ]:
from pathlib import Path
from itertools import product
from functools import lru_cache
import hashlib, importlib.metadata, platform, re
import numpy as np
import pandas as pd
from scipy.linalg import qr
from scipy.stats import t as student_t
import statsmodels.api as sm
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from threadpoolctl import threadpool_limits

# 避免小矩阵运算被过多 BLAS 线程拖慢；不修改系统环境。
blas_limit = threadpool_limits(limits=1)

def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'code' / 'audit_lib.py').is_file():
            return p
    raise FileNotFoundError('Open the notebook from this repository.')

ROOT = find_root()
ORIGINAL = ROOT / 'data' / 'raw'
import sys
sys.path.insert(0, str(ROOT / 'code'))
pd.set_option('display.max_columns', 12)
pd.set_option('display.precision', 6)
from matplotlib import font_manager
plot_font = 'Times New Roman' if any(f.name == 'Times New Roman' for f in font_manager.fontManager.ttflist) else 'DejaVu Serif'
plt.rcParams.update({'font.family': plot_font, 'mathtext.fontset': 'stix',
                     'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.titlesize': 15,
                     'axes.labelsize': 12, 'xtick.labelsize': 11, 'ytick.labelsize': 11})
COLORS = {'long': '#7CB6D3', 'short': '#8B1E2D', 'none': '#222222', 'muted': '#70869B'}

In [ ]:
import json
manifest=json.loads((ROOT / 'data' / 'sources.json').read_text(encoding='utf-8'))
EXPECTED_SHA256={x['filename']:x['sha256'] for x in manifest['files']}
source_checks = []
for name, expected in EXPECTED_SHA256.items():
    p = ORIGINAL / name
    actual = hashlib.sha256(p.read_bytes()).hexdigest()
    assert actual == expected, f'文件已变化，请检查来源：{name}'
    source_checks.append({'文件': name, '字节': p.stat().st_size, 'SHA256': actual})
display(pd.DataFrame(source_checks))
import sys
print('Python', platform.python_version())
print('解释器：', sys.executable)
print('虚拟环境：', sys.prefix != sys.base_prefix)
print({p: importlib.metadata.version(p) for p in ['numpy', 'pandas', 'scipy', 'statsmodels', 'matplotlib']})

### 2.1 加载数据，保留原始处理编码

一行是一场议会会议；`particularistic` 是作者归为定向立法的主题份额总和，单位为 0–1 比例。回归系数乘 100 后才是**百分点**。

`treat4=1` 为长期激励、`treat2=1` 为短期激励；墨西哥城 `CdMex` 和普埃布拉 `Puebla` 的两个变量均为 0。其他比较中不适用的组被作者设为缺失值；**不能把这些 NA 填成 0**。

`reform` 沿用作者已编码的议会任期，不能统一改成“2018 年后”。它在不同州可能于不同时间取 1，日期编码也有边界记录。作者 `date2` 的实际构造是 `realyear*100` 后再拼接两位月份，即 `年份*10000+月份`。下面保留这一编码以直接执行原脚本日期条件；另建正常日期仅用于阅读和画图。

In [ ]:
raw = pd.read_csv(ORIGINAL / 'allres.csv')
assert raw.shape == (6890, 471)
data = raw.copy()
date = pd.to_datetime(data.realyear.astype(str) + ' ' + data.month,
                      format='%Y %B', errors='coerce')
data['date'] = date
data['date2'] = date.dt.year * 10000 + date.dt.month
data['month_index'] = date.dt.year * 12 + date.dt.month
data['D_long'] = data.treat4 * data.reform
data['D_short'] = data.treat2 * data.reform
data['treat42'] = np.select([data.treat4.eq(1), data.treat2.eq(1)], [1., 0.], default=np.nan)
data['D_horizon'] = data.treat42 * data.reform
topic_columns = [f'V{i}' for i in range(1, 451)]
assert set(topic_columns).issubset(data.columns)
assert data.state.nunique() == 20
display(pd.DataFrame({'字段': ['state', 'date', 'particularistic', 'treat4', 'treat2', 'reform'],
                      '缺失行数': [data[c].isna().sum() for c in
                                 ['state', 'date', 'particularistic', 'treat4', 'treat2', 'reform']]}))
display(data[['state', 'realyear', 'month', 'treat4', 'treat2', 'reform', 'particularistic']].head())

In [ ]:
def group_label(row):
    if row.treat4 == 1:
        return '长期激励'
    if row.treat2 == 1:
        return '短期激励'
    return '无连任激励'

data['group'] = data.apply(group_label, axis=1)
group_inventory = data.groupby(['group', 'state']).agg(会议数=('state', 'size'),
                  首次记录=('date', 'min'),末次记录=('date', 'max')).reset_index()
display(group_inventory)
display(data.groupby('group').agg(州数=('state', 'nunique'), 会议数=('state', 'size')))

### 2.2 检查 CTM 份额如何变成因变量

作者 R 脚本使用 `stm(alldfm, K=450, seed=12345678)`，随后按分类表将 54 个主题相加。以下主题编号来自作者公开的 [`particularistic.csv`](https://dataverse.harvard.edu/api/access/datafile/3892157?format=original)，由获取脚本从官方来源下载到 `data/raw/`，本 notebook 读取下载文件，不嵌入分类表。

这是对**已估计主题份额及其汇总**的检查，不是重新拟合 CTM，也没有将主题估计不确定性传播到回归标准误。

对阅读报告中主题数的表述，应区分“作者选择了 450 个主题”与“我们独立证明 450 是最小充分设定”。现有 notebook 不比较不同 K，也不支持“更多主题没有意义”的判断。图2、图3的测量验证不能直接验证后续 DID 的因果识别。

In [ ]:
particularistic_topics = pd.read_csv(ORIGINAL / 'particularistic.csv')['part'].tolist()
assert len(set(particularistic_topics)) == 54
rebuilt_y = raw[[f'V{i}' for i in particularistic_topics]].sum(axis=1)
max_y_gap = (rebuilt_y - raw.particularistic).abs().max()
topic_sum_gap = (raw[topic_columns].sum(axis=1) - 1).abs().max()
# 明确采用比例单位的数值容差，不要求 CSV 导出的各列逐位相等。
assert np.allclose(rebuilt_y, raw.particularistic, atol=1e-6, rtol=0)
assert np.allclose(raw[topic_columns].sum(axis=1), 1, atol=1e-5, rtol=0)
display(pd.DataFrame({'检查': ['450 个主题份额总和与 1 的最大差',
                              '54 个主题加总与原因变量的最大差'],
                      '数值': [topic_sum_gap, max_y_gap]}))

CSV 中逐列保存的份额与作者已汇总的因变量并非逐位相等：本数据的最大汇总差约为 $4.94\times10^{-7}$ 个比例单位（约 0.0000494 个百分点），450 个主题之和偏离 1 的最大值约为 $1.615\times10^{-6}$。这与小量数值导出精度差异相容；回归完整沿用作者 `particularistic` 列，不用重新汇总值替换。

## 3. 设定对照与基准复现

### 3.1 样本、模型和软件处理是否一致

原文模型的核心是

$$Y_{ist}=\alpha+\beta D_{st}+\mu_s+\gamma_t+\varepsilon_{ist}.$$

$\mu_s$ 为州固定效应，$\gamma_t$ 为年月固定效应；$D$ 由作者的分组和 `reform` 编码相乘得到。此处写为 $T_{st}$ 可以提醒读者：`reform` 所表示的任期状态随州而异，不是全体共同的“2018年后”。

下面从原文件打印作者命令，并列出 Python 对照。我们先核对同一模型，不通过改用其他固定效应来判断原规格是否存在问题。

In [ ]:
author_code = (ORIGINAL / 'replication_analysis.do').read_text(encoding='utf-8')
for marker in ['*Table 2: H1', '*Table 3: H3']:
    block = author_code[author_code.index(marker):].splitlines()
    command = next(line.strip() for line in block if line.strip().startswith('reg '))
    print(marker, '\n', command, '\n')
scope_comparison = pd.DataFrame([
    {'项目':'H1','原文位置':'Table 2, p.108','原处理项':'timb3=treat4*reform',
     'Python处理项':'D_long','样本规则':'treat4适用组，保留原NA排除'},
    {'项目':'H3','原文位置':'Table 3, p.111','原处理项':'c.treat42#c.reform',
     'Python处理项':'D_horizon','样本规则':'长期与短期组'},
    {'项目':'H2末年','原文位置':'Figure 6, p.110','原处理项':'timb3，if timingc==1',
     'Python处理项':'D_long + 原timingc筛选','样本规则':'从.do逐条解析6条replace条件'},
    {'项目':'补充比较','原文位置':'我们新增','原处理项':'正文未直接报告此比较',
     'Python处理项':'D_short','样本规则':'短期与无连任组，直接重新估计'}
])
display(scope_comparison)
display(pd.DataFrame([
    {'设置':'固定效应','作者':'州 + 年月','Python':'同一固定效应空间'},
    {'设置':'标准误','作者':'按州聚类','Python':'州聚类CR1，t(G−1)参考分布'},
    {'设置':'数据与因变量','作者':'allres.csv / particularistic','Python':'保留原值及处理编码'},
    {'设置':'完全共线性','作者':'Stata可自动省略冗余项','Python':'先检查残差变化与矩阵秩；未识别时不输出效应'},
    {'设置':'额外bootstrap','作者':'此处不冒充原文程序','Python':'仅用于短期组补充比较，另行标明'}
]))

### 3.2 Python 估计实现

下方用 Frisch–Waugh–Lovell 残差化估计；QR 分解仅保留固定效应的独立列。完全共线时返回“未识别”，不会让伪逆自动给出一个被误解为独立效应的系数。

标准误采用州聚类 CR1，常规 p 值和 95% 区间使用 $t(G-1)$。少聚类补充推断使用零假设约束下的 Rademacher wild cluster bootstrap，枚举全部 $2^G$ 符号组合。原始 Stata 文件保留在目录中，作为设定对照。

In [ ]:
from audit_lib import FixedEffectsDesign
def design_for(column, frame=data):
    return FixedEffectsDesign(frame.dropna(subset=[column]))

def result_table(records):
    table = pd.DataFrame(records).copy()
    for source, target in [('b','估计（百分点）'), ('se','聚类SE（百分点）'),
                            ('ci_lower','95%下限'), ('ci_upper','95%上限')]:
        table[target] = table[source]*100
    return table[['comparison','n','states','identified','估计（百分点）','聚类SE（百分点）',
                  '95%下限','95%上限','p','wild_p','rank_FE','rank_FE_D']].rename(
                  columns={'comparison':'比较','n':'会议数','states':'州数','identified':'独立识别',
                           'p':'常规p','wild_p':'wild bootstrap p'})

def forest(ax, records, labels, title, colors):
    for i, (r, label, color) in enumerate(zip(records, labels, colors)):
        if r['identified']:
            ax.errorbar(r['b']*100, i,
                xerr=[[100*(r['b']-r['ci_lower'])], [100*(r['ci_upper']-r['b'])]],
                fmt='o', color=color, capsize=4, markersize=7)
    ax.axvline(0, color='#555555', linestyle='--', linewidth=1)
    ax.set_yticks(range(len(labels)), labels)
    ax.set_ylim(len(labels)-.5, -.5)
    ax.set_xlabel('DID coefficient (percentage points); 95% clustered CI')
    ax.set_title(title, loc='left', pad=14)
    ax.grid(axis='x', color='#E4EAF0', linewidth=.7)

### 3.3 H1、H3 基准复现

先核对原文主要点估计与处理项的独立变化，再审计 H2。H1/H3 与 H2 使用同一类州、年月固定效应框架，但比较组和估计样本不同。两个基准系数不能直接相减来代替短期组与无连任组的重新估计。

In [ ]:
h1_design = design_for('D_long')
h3_design = design_for('D_horizon')
h1 = h1_design.fit('D_long'); h1['comparison'] = 'H1：长期 vs 无连任'
h3 = h3_design.fit('D_horizon'); h3['comparison'] = 'H3：长期 vs 短期'
assert h1['n'] == 4899 and np.isclose(h1['b'], .01525336444865884)
assert h3['n'] == 6062 and np.isclose(h3['b'], .02623265672428348)
display(result_table([h1, h3]))

# 用 statsmodels 的完整 OLS + 聚类协方差交叉核对 FWL 实现。
X_check = np.column_stack([h1_design.controls, h1_design.frame.D_long])
ols_check = sm.OLS(h1_design.y, X_check).fit(cov_type='cluster',
    cov_kwds={'groups': h1_design.frame.state}, use_t=True)
assert np.isclose(ols_check.params[-1], h1['b'], atol=1e-12)
assert np.isclose(ols_check.bse[-1], h1['se'], atol=1e-12)
fig, ax = plt.subplots(figsize=(10, 3.3), layout='constrained')
forest(ax, [h1,h3], ['H1: Long-term vs term-limited','H3: Long-term vs short-term'],
       'Baseline replication', [COLORS['long'], COLORS['short']])
plt.show()

H1 约为 **+1.53 个百分点**。按本 notebook 明确采用的 CR1 和 $t(G-1)$ 计算，精确 p 值约 **0.050008**，紧邻 0.05；不要仅因四舍五入的 `0.050` 写成确定小于 0.05。H3 约为 **+2.62 个百分点**。两个基准回归的处理项均有固定效应之外的独立变化。

### 3.4 原文 H1/H3 的处理前检查

以下从原始 `.do` 的 `rm101` 日期条件提取各州的选举年份，再按原脚本顺序构造 12+12+9 个切点。每个切点是一次单独的回归，不是一个包含全部 leads 的事件研究。置信区间为逐项常规 95% 区间。

In [ ]:
author_code = (ORIGINAL / 'replication_analysis.do').read_text(encoding='utf-8')
cohort = {}
for threshold, expression in re.findall(
        r'(?:gen|replace) rm10[^=]*=0 if date2<(\d+)[^&]*& \(([^\r\n]*)\)', author_code):
    for state in re.findall(r'state=="([^"]+)"', expression):
        cohort[state] = int(threshold[:4])
assert len(cohort) == 20 and cohort['Puebla'] == 2013 and cohort['CdMex'] == 2015
cutoffs = [(shift, month) for shift, count in [(2,12),(1,12),(0,9)]
                         for month in range(1,count+1)]

def placebo_checks(group_column, wild=False):
    sample = data.loc[data.reform.eq(0) & data[group_column].notna()].copy()
    design = FixedEffectsDesign(sample)
    records, draws = [], []
    for i, (shift, month) in enumerate(cutoffs, 1):
        threshold = (design.frame.state.map(cohort)-shift)*10000+month
        x = design.frame[group_column] * design.frame.date2.ge(threshold)
        r = design.fit(x, wild=wild, return_boot=wild)
        if wild:
            draws.append(r.pop('boot_t'))
        r.update(cutoff=i, cohort_shift=shift, month=month)
        records.append(r)
    table = pd.DataFrame(records)
    joint_p = np.nan
    if wild:
        # 所有切点使用同一套、同一州顺序的枚举权重，保留检验间依赖。
        observed = np.max(np.abs(table.b/table.se))
        joint_p = float(np.mean(np.max(np.vstack(draws),axis=0) >= observed-1e-12))
    return table, joint_p

pre_h1, _ = placebo_checks('treat4')
pre_h3, _ = placebo_checks('treat42')
display(pd.DataFrame({'比较':['H1：长期 vs 无连任','H3：长期 vs 短期'],
    '切点数':[len(pre_h1),len(pre_h3)],
    '常规p<0.05个数':[(pre_h1.p<.05).sum(),(pre_h3.p<.05).sum()]}))

fig, axes = plt.subplots(2,1,figsize=(11,6),sharex=True,layout='constrained')
for ax, table, title in zip(axes,[pre_h1,pre_h3],
                 ['H1: Long-term vs term-limited','H3: Long-term vs short-term']):
    ax.errorbar(table.cutoff,table.b*100,
        yerr=np.vstack([(table.b-table.ci_lower)*100,(table.ci_upper-table.b)*100]),
        fmt='s',ms=4,capsize=2,color=COLORS['muted'])
    ax.axhline(0,color='black',ls='--',lw=1)
    ax.set_title(title,loc='left');ax.set_ylabel('Estimate (pp)')
axes[-1].set_xlabel('Placebo cutoff in original script order (33 separate regressions)')
axes[-1].set_xticks([1,6,12,18,24,30,33])
plt.show()

处理前检查应结合方向、形状、功效与多重检验阅读。本次 H1 常规逐项检验有 4 个 p<0.05，H3 为 0；不因为偏离方向为负，就把它自动视为无关。未发现持续正向提前效应，与证明平行趋势成立是两种不同判断。这里的结果也不与后文短期组的 6 个显著切点混用。

## 4. 主审计：H2 末年估计的独立识别

审计问题是：**控制州和年月固定效应后，末年的处理变量还剩下多少独立变化？**这一步检查处理效应在给定规格下是否能单独估计；不把“显著”或“有软件输出”当成识别的替代。

### 4.1 重建作者的三个周期子样本

从 `.do` 直接提取 `timinga/timingb/timingc` 条件，对应 Figure 6 的 `Election (t)`、`t+1`、`t+2`。作者按相对选举周期筛选日历窗口，不能用“全部州都严格三年一届”代替其实际日期条件。

In [ ]:
start = author_code.index('*Figure 6: H2')
end = author_code.index('coefplot', start)
print(author_code[start:end].strip())

def original_timing_mask(name):
    clauses = re.findall(r'replace '+name+r'=1 if date2>(\d+) & date2<=(\d+) & \(([^\r\n]*)\)',
                         author_code)
    assert len(clauses) == 6
    mask = pd.Series(False, index=data.index)
    for lo, hi, expression in clauses:
        states = re.findall(r'state=="([^"]+)"', expression)
        mask |= data.date2.gt(int(lo)) & data.date2.le(int(hi)) & data.state.isin(states)
    return mask

timing = {name: original_timing_mask(name) for name in ['timinga','timingb','timingc']}
h2_records, h2_designs = [], {}
for name, label in [('timinga','Election (t)'),('timingb','t+1'),('timingc','t+2')]:
    model = FixedEffectsDesign(data.loc[timing[name] & data.D_long.notna()])
    h2_designs[name] = model
    r = model.fit('D_long');r['comparison'] = label
    h2_records.append(r)
display(result_table(h2_records))
assert h2_records[2]['n'] == 1511 and not h2_records[2]['identified']

表中 `t+2` 的系数留空，表示在保留全部固定效应空间时处理项没有独立变化。下面先解释样本，再说明为什么添加额外省略约束仍可复现图中约 4.60 的数值。

### 4.2 样本覆盖：关键比较组合在哪里缺失

In [ ]:
h2 = h2_designs['timingc'].frame.copy()
h2['D'] = h2.D_long
h2['L'] = h2.date2.between(20170007,20180006).astype(int)
h2['E'] = h2.date2.between(20120007,20130006).astype(int)
h2['P'] = h2.state.eq('Puebla').astype(int)
windows = [(20120007,20130006),(20140007,20150006),
           (20150007,20160006),(20170007,20180006)]
window_labels = ['Jul 2012–Jun 2013','Jul 2014–Jun 2015',
                 'Jul 2015–Jun 2016','Jul 2017–Jun 2018']
group_masks = [h2.treat4.eq(1),h2.state.eq('CdMex'),h2.state.eq('Puebla')]
coverage = pd.DataFrame([[int((mask & h2.date2.between(lo,hi)).sum())
                          for lo,hi in windows] for mask in group_masks],
                          index=['Long-term','Mexico City','Puebla'],columns=window_labels)
assert coverage.to_numpy().tolist() == [[0,491,346,492],[0,81,0,0],[69,0,0,32]]
display(coverage)
fig, ax = plt.subplots(figsize=(11,3.5),layout='constrained')
ax.imshow(coverage, cmap='Blues', vmin=0, vmax=coverage.to_numpy().max(),aspect='auto')
for i in range(3):
    for j in range(4):
        n = int(coverage.iloc[i,j])
        ax.text(j,i,str(n) if n else '—',ha='center',va='center',fontsize=17,
                color='white' if n>300 else '#172A3D')
ax.set_xticks(range(4),[s.replace('–','–\n') for s in window_labels])
ax.set_yticks(range(3),coverage.index)
ax.set_title('H2 final-year sample: Sessions by selected calendar window (— = no observations)',loc='left')
plt.show()

cdmex_coding = h2.loc[h2.state.eq('CdMex')].groupby('reform').size().rename('会议数')
display(cdmex_coding)

末期只有普埃布拉提供无连任对照。墨西哥城 81 条较早窗口记录中有 1 条 `reform=1`，不重新改写作者编码。普埃布拉用于较早比较的记录在 2012/13，缺少中间日历窗口。缺的是原估计子样本的关键州—年月组合，不只是总量小，也不等于这些州在原始数据库中完全没有其他时期记录。

### 4.3 先对照正常 DID：为什么标准2×2的处理项不能被吸收

这里计算的是**处理变量 D 的双重差分**，用于检查独立变化，不是结果变量 Y 的双重差分。对图示的标准两组、两期处理状态 $\begin{pmatrix}0&1\\0&0\end{pmatrix}$，有 $(1-0)-(0-0)=1$。

假设 $D_{st}=a_s+b_t$：$a_s$ 是同一州不随时间改变的组成部分；$b_t$ 是同一年月对各州共同的组成部分。则两组都只能变化 $b_1-b_0$，二者变化之差必须为0。标准处理状态给出1，故无法这样表示。

在原 H2 末年样本中，却可以给出一套 $a_s,b_t$ 满足全部已有观测：长期组和墨西哥城的 $a_s=0$，普埃布拉的 $a_s=-1$；早期、末期的 $b_t=1$，中间窗口的 $b_t=0$。这些数值是对 D 的表示，不是回归 Y 得到的州效应与时间效应。

In [ ]:
S = np.array([1.,1.,0.,0.]); T = np.array([0.,1.,0.,1.]); D = S*T
F_standard = np.column_stack([np.ones(4),S,T])
standard_residual = D-F_standard@np.linalg.lstsq(F_standard,D,rcond=None)[0]
assert np.isclose(standard_residual@standard_residual,.25)
display(pd.DataFrame([
 {'比较':'标准2×2示例','FE秩':np.linalg.matrix_rank(F_standard),
  'FE+D秩':np.linalg.matrix_rank(np.column_stack([F_standard,D])),
  '残差化D平方和':standard_residual@standard_residual},
 {'比较':'原H2末年样本','FE秩':59,'FE+D秩':59,
  '残差化D平方和':h2_records[2]['residual_D_ss']}
]))

In [ ]:
# 由本次运行的实际 h2 样本生成右图；缺失组合保持 NaN，不补成0。
from matplotlib.colors import ListedColormap, BoundaryNorm
display_frame = h2.assign(
    display_group=np.select([h2.treat4.eq(1),h2.state.eq('CdMex')],
                            ['Long-term','Mexico City'],default='Puebla'),
    calendar=np.select([h2.E.eq(1),h2.L.eq(1)],['2012/13','2017/18'],default='2014–16'))
assert display_frame.groupby(['display_group','calendar']).D.nunique().le(1).all()
observed_matrix = display_frame.pivot_table(index='display_group',columns='calendar',
                      values='D',aggfunc='first').reindex(
                      index=['Long-term','Mexico City','Puebla'],
                      columns=['2012/13','2014–16','2017/18'])
assert np.array_equal(h2.D, -h2.P + (h2.L+h2.E))
cmap = ListedColormap(['#EDF1F4','#7CB6D3']); cmap.set_bad('#FFFFFF')
fig, axes = plt.subplots(1,2,figsize=(12,4.8),layout='constrained')
panels = [(np.array([[0.,1.],[0.,0.]]),['Treatment','Control'],['Before','After'],
           'Standard DID: Independent treatment variation'),
          (observed_matrix.to_numpy(),['Long-term (a=0)','Mexico City (a=0)','Puebla (a=−1)'],
           ['2012/13\nb=1','2014–16\nb=0','2017/18\nb=1'],
           'H2 final-year sample: D is additive')]
for ax, (values, row_names, column_names, title) in zip(axes,panels):
    ax.imshow(np.ma.masked_invalid(values),cmap=cmap,
              norm=BoundaryNorm([-.5,.5,1.5],2),aspect='auto')
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            val=values[i,j]
            ax.text(j,i,'—' if np.isnan(val) else str(int(val)),ha='center',va='center',fontsize=22)
    ax.set_xticks(range(len(column_names)),column_names)
    ax.set_yticks(range(len(row_names)),row_names)
    ax.set_title(title,loc='left',fontsize=13,pad=12)
axes[0].set_xlabel(r'$(1-0)-(0-0)=1\ne0$',fontsize=18,labelpad=13)
axes[1].set_xlabel(r'$D_{st}=a_s+b_t,\quad\widetilde D=0$',fontsize=18,labelpad=13)
plt.show()

### 4.4 逐行恒等式与矩阵秩：原子样本的直接证据

定义 $L_t$ 为2017年7月至2018年6月窗口，$E_t$ 为2012年7月至2013年6月窗口，$P_s$ 为普埃布拉指标。在这个子样本中：

$$D_{st}=R_sT_{st}=L_t+E_t-P_s.$$

$L_t,E_t$ 可由年月虚拟变量表示，$P_s$ 可由州虚拟变量表示。因此 $D$ 属于固定效应张成的空间。下面逐行核对并比较加入 D 前后的秩。处理项有两种写法本身不是问题；第二种写法完全由现有控制项组成，才构成这里的线性依赖。

In [ ]:
identity_error = h2.D - (h2.L+h2.E-h2.P)
assert (identity_error == 0).all()
identity_rows = h2.groupby(['group','reform','L','E','P'],dropna=False).agg(
    N=('D','size'),D=('D','first')).reset_index()
identity_rows['L+E-P'] = identity_rows.L+identity_rows.E-identity_rows.P
display(identity_rows)
display(pd.DataFrame([
    {'比较':'H1 整体样本','N':h1['n'],'FE秩':h1['rank_FE'],'FE+D秩':h1['rank_FE_D'],
     '残差化D平方和':h1['residual_D_ss']},
    {'比较':'H2 末年样本','N':len(h2),'FE秩':h2_records[2]['rank_FE'],
     'FE+D秩':h2_records[2]['rank_FE_D'],'残差化D平方和':h2_records[2]['residual_D_ss']}
]))

### 4.5 同一拟合可以容纳不同β：量化识别后果

代入原模型：

$$Y=\alpha+\beta D+\gamma_t+\mu_s+\varepsilon
=\alpha+[\gamma_t+\beta(L_t+E_t)]+[\mu_s-\beta P_s]+\varepsilon.$$

因此改变 $\beta$，并相应调整固定效应，预测可以不变。下面用同一个样本验证：指定的 $\beta$ 是识别诊断情景，不是四次独立估计出的效应。

In [ ]:
def explicit_fe(frame):
    states = pd.get_dummies(frame.state, prefix='s',drop_first=True,dtype=float)
    months = pd.get_dummies(frame.date2.astype(int), prefix='m',drop_first=True,dtype=float)
    return pd.concat([pd.Series(1.,index=frame.index,name='const'),states,months],axis=1)

F_h2 = explicit_fe(h2)
assert np.linalg.matrix_rank(F_h2) == F_h2.shape[1] == 59
pure_fe = sm.OLS(h2.particularistic,F_h2).fit()
same_fit = []
for specified_beta in [-.10,0,.04599196022827533,.10]:
    nuisance = np.linalg.lstsq(F_h2, h2.particularistic-specified_beta*h2.D,rcond=None)[0]
    prediction = F_h2 @ nuisance + specified_beta*h2.D
    gap = np.max(np.abs(prediction-pure_fe.fittedvalues))
    assert gap < 1e-12
    same_fit.append({'指定β（百分点）':specified_beta*100,
                     '预测值最大差':gap,
                     '残差平方和':float(np.sum((h2.particularistic-prediction)**2))})
display(pd.DataFrame(same_fit))

### 4.6 4.60与5.58：额外省略约束下的参数化

共同基准为 **2012年7月**与 **Campeche**。正常省略基准组后，固定效应矩阵已经满列秩；再保留 D 并额外省略一个中间日历窗口的月份项，就施加了额外约束。这与普通的基准组省略不同。

以下报告的是额外约束下的系数和SE，不称为完整规格中独立识别的改革效应。上三行数值构成反对称关系，不能称为“反对角矩阵”。

为核对数值来源，我们另作顺序保留独立列的重构：保留截距、D、州项和升序年月项时，2016年6月成为冗余列。它不是全部月份中的最后一项；我们未执行原生 Stata、未取得作者日志，不把这项重构表述为已观察到的作者运行记录。

In [ ]:
constrained_models = {}
allocation = {'纯固定效应': {'beta':np.nan, **pure_fe.params.to_dict()}}
for omitted, label in [(20160006,'额外省略2016年6月'),(20140007,'额外省略2014年7月')]:
    X = pd.concat([F_h2[['const']],h2.D.rename('D'),F_h2.drop(columns=['const',f'm_{omitted}'])],axis=1)
    assert np.linalg.matrix_rank(X) == X.shape[1] == 59
    fit = sm.OLS(h2.particularistic,X).fit(cov_type='cluster',
        cov_kwds={'groups':h2.state},use_t=True)
    constrained_models[label] = fit
    parameters = fit.params.to_dict()
    parameters[f'm_{omitted}'] = 0.
    parameters['beta'] = parameters.pop('D')
    allocation[label] = parameters
    assert np.max(np.abs(fit.fittedvalues-pure_fe.fittedvalues)) < 1e-12

coefficient_rows = [('β','beta'),('2016年6月 γ','m_20160006'),
                    ('2014年7月 γ','m_20140007'),('普埃布拉 μ','s_Puebla'),('截距 α','const')]
allocation_table = pd.DataFrame({name:{label:values.get(key,np.nan)*100
                      for label,key in coefficient_rows} for name,values in allocation.items()})
display(allocation_table.rename_axis('参数（百分点）'))
constraint_inference = []
for label, fit in constrained_models.items():
    constraint_inference.append({'额外约束':label,'β（百分点）':fit.params.D*100,
        'SE（百分点）':np.sqrt(fit.cov_params().loc['D','D'])*100,'p（约束下）':2*student_t.sf(abs(fit.params.D/np.sqrt(fit.cov_params().loc['D','D'])),h2.state.nunique()-1),
        '与纯FE预测的最大差':np.abs(fit.fittedvalues-pure_fe.fittedvalues).max()})
display(pd.DataFrame(constraint_inference))
assert np.isclose(constrained_models['额外省略2016年6月'].params.D,.04599196022827533)
assert np.isclose(constrained_models['额外省略2014年7月'].params.D,.05577290025185785)

In [ ]:
# 仅演示与作者命令变量顺序一致的“顺序保留独立列”重构。
candidate = pd.concat([F_h2[['const']],h2.D.rename('D'),F_h2.drop(columns='const')],axis=1)
basis = []
omitted_columns = []
for column in candidate.columns:
    v = candidate[column].to_numpy(float)
    residual = v.copy()
    for _ in range(2):  # 再正交化，降低浮点误差。
        for q in basis:
            residual -= q*(q@residual)
    norm = np.linalg.norm(residual)
    if norm < 1e-8:
        omitted_columns.append(column)
    else:
        basis.append(residual/norm)
assert omitted_columns == ['m_20160006']
print('顺序重构的额外冗余列：', omitted_columns)

**正负号的来源。**在上述共同基准下，对中间月份 $L_t=E_t=0$，两种参数化有

$$\gamma_t^{FE}=\gamma_t-\beta.$$

额外省略 2016 年 6 月即约束 $\gamma_{2016.6}=0$，因此

$$\gamma_{2016.6}^{FE}=-0.04599196=-\beta
\quad\Rightarrow\quad\beta=+0.04599196.$$

这不是“月份导致改革效应”，而是同一拟合在不同参数约束下的系数分配。更小的标准误或 bootstrap 不能创造缺失的独立处理变化。

### 4.7 补充记录的设计检查：定位缺失信息，不冒充修复后的H2

两种检查必须分开：墨西哥城末期记录是**假设观测，仅检验设计矩阵，不模拟结果**；普埃布拉 2014 年 7 月记录在原始文件中真实存在，只是被 `timingc` 排除。

补回后能打破共线性，不等于已修复原来的 H2：改变筛选会改变相对选举周期位置和估计样本。本节只定位识别信息如何随筛选变化。

In [ ]:
def rank_check(frame):
    X = explicit_fe(frame)
    return int(np.linalg.matrix_rank(X)), int(np.linalg.matrix_rank(np.column_stack([X,frame.D])))

minimal = h2[['state','date2','D']].copy()
hypothetical = pd.concat([minimal,pd.DataFrame([{'state':'CdMex','date2':20170007,'D':0.}])],ignore_index=True)
puebla_actual = data.loc[data.state.eq('Puebla') & data.date2.eq(20140007)].copy()
puebla_actual['D'] = puebla_actual.D_long
assert len(puebla_actual) == 5 and puebla_actual.D.eq(0).all()
restored = pd.concat([minimal,puebla_actual[['state','date2','D']]],ignore_index=True)
checks = [
    {'样本':'原 H2 末年样本','N':len(minimal),'新增记录类型':'无','FE秩':rank_check(minimal)[0],'FE+D秩':rank_check(minimal)[1]},
    {'样本':'加入墨西哥城2017年7月','N':len(hypothetical),'新增记录类型':'1 条假设设计记录，无Y',
     'FE秩':rank_check(hypothetical)[0],'FE+D秩':rank_check(hypothetical)[1]},
    {'样本':'补回普埃布拉2014年7月','N':len(restored),'新增记录类型':'5 条真实记录',
     'FE秩':rank_check(restored)[0],'FE+D秩':rank_check(restored)[1]}
]
assert rank_check(hypothetical)==(59,60) and rank_check(restored)==(59,60)
display(pd.DataFrame(checks))
display(pd.DataFrame([
 {'新增单元':'墨西哥城2017年7月（假设）','R×T':0,'L+E-P':1},
 {'新增单元':'普埃布拉2014年7月（真实）','R×T':0,'L+E-P':-1}]))

**H2主审计到此的结论。**在原脚本实际保留的1511条记录上，D可以由固定效应精确表示；正常设定基准组后，再加D不增加秩。改变β并相应调整固定效应，预测不变。因此质疑针对原文 Figure 6 的末年估计在所述完整规格下的独立识别。补回其他周期位置的记录会改变分析样本，不能直接作为原H2已修复的证据。

## 5. 补充分析：短期连任 vs 无连任

这一节另起一条证据线，不用于证明H2共线性。原文 Table 2 比较长期与无激励，Table 3 比较长期与短期；这里直接估计短期与无激励的变化差异，不能用不同样本下两个原文系数相减来代替。

以 `treat2 × reform` 为处理项，沿用原始编码、州和年月固定效应。样本含6个短期州和2个无连任地区，共8个聚类；少聚类和对照来源少都影响推断，不能只说“有6个州”。

### 5.1 点估计、少聚类推断与对照地区剔除

In [ ]:
short_results = []
for label, excluded in [('完整样本',None),('剔除墨西哥城','CdMex'),('剔除普埃布拉','Puebla')]:
    frame = data.loc[data.D_short.notna()].copy()
    if excluded:
        frame = frame.loc[frame.state.ne(excluded)]
    model = FixedEffectsDesign(frame)
    r = model.fit('D_short', wild=True)
    r['comparison'] = label
    short_results.append(r)
assert short_results[0]['n'] == 2807
assert np.isclose(short_results[0]['b'], -.019597792144748865)
assert short_results[0]['wild_p'] == .0625
display(result_table(short_results))
fig, ax = plt.subplots(figsize=(10,4),layout='constrained')
forest(ax, short_results, ['Full sample','Exclude Mexico City','Exclude Puebla'],
       'Extension: Short-term vs term-limited', [COLORS['short']]*3)
plt.show()

完整样本的点估计约 **−1.96个百分点**。常规州聚类p值约0.0019，枚举wild cluster bootstrap p值为 **0.0625**；剔除任一对照地区后，bootstrap证据更弱。因此报告点估计与完整推断，既不只展示显著的常规p值，也不因为bootstrap不显著就断言没有效应。

### 5.2 处理前切点与多重比较

沿用相同33个切点，仅使用改革前记录。若每次检验都具有正确的5%误报率，在全部零假设成立时，预期显著数为33×0.05=1.65；但切点使用高度重叠的数据，彼此相关，不能把显著个数当成独立二项分布来计算整体证据。

下面采用保留共同州顺序与权重的联合bootstrap，检验最大绝对t统计量。它检验整体偏离，不是直接给“出现6个显著结果”的次数事件计算概率。

In [ ]:
short_placebos, short_joint_p = placebo_checks('treat2', wild=True)
significant = short_placebos.p.lt(.05)
assert len(short_placebos) == 33 and significant.sum() == 6
assert (short_placebos.loc[significant,'b'] < 0).all()
assert np.isclose(short_joint_p, .1171875)
display(pd.DataFrame({'切点数':[33], '常规显著个数':[int(significant.sum())],
                       '联合max-|t| bootstrap p':[short_joint_p]}))
display(short_placebos[['cutoff','cohort_shift','month','b','se','p','wild_p']].rename(
    columns={'cutoff':'切点序号','cohort_shift':'年份前移','month':'月份',
             'b':'系数（比例单位）','se':'聚类SE','p':'常规p','wild_p':'bootstrap p'}))
fig, ax = plt.subplots(figsize=(11,4),layout='constrained')
for _, row in short_placebos.iterrows():
    color = COLORS['short'] if row.p < .05 else COLORS['muted']
    ax.errorbar(row.cutoff,row.b*100,
        yerr=[[100*(row.b-row.ci_lower)],[100*(row.ci_upper-row.b)]],
        fmt='o' if row.p<.05 else 's',ms=4,capsize=2,color=color)
ax.axhline(0,color='black',ls='--',lw=1)
ax.set(xlabel='Placebo cutoff (original order)',ylabel='DID estimate (pp)',
       xticks=[1,6,12,18,24,30,33])
ax.set_title(f'Short-term vs term-limited: Pre-treatment checks; joint p = {short_joint_p:.3f}',loc='left')
plt.show()

6个切点在常规逐项检验中显著，方向均为负；联合最大绝对t检验p≈0.117，未在常用水平下拒绝共同为零，也不等于证明平行趋势成立。对负向处理后点估计，负向处理前变化尤其需要一起呈现，不能宣称补充回归已经确立负向因果效应。

## 6. 影响评估与方法问答

### 6.1 H2受影响，是否意味着H1/H3也失效

三个检验使用同一类固定效应框架，但比较组与样本不同。是否具有独立处理变化，需要逐个样本检查；H1/H3的点估计可复现，不等于其全部因果识别假设已被验证。下面将本次检查覆盖的范围列清楚。

In [ ]:
impact_rows = []
for name, r, conclusion in [
    ('H1：长期 vs 无连任',h1,'点估计可复现，未发现相同的完全共线性；仍需识别假设'),
    ('H2：末年',h2_records[2],'完整固定效应规格下不能单独识别末年处理系数'),
    ('H3：长期 vs 短期',h3,'点估计可复现，未发现相同的完全共线性；不决定短期组相对无激励的方向')]:
    impact_rows.append({'论断':name,'N':r['n'],
        '点估计（百分点）':r['b']*100 if r['identified'] else np.nan,
        'FE秩':r['rank_FE'],'FE+D秩':r['rank_FE_D'],'本次判断':conclusion})
display(pd.DataFrame(impact_rows))

### 6.2 方法问答

**为什么有显著系数，还可能缺乏独立识别？**软件可以通过省略冗余列完成计算。特定额外约束下报告出的系数与SE，不自动等于完整模型中的独立处理效应。这里的直接证据是D完全落在固定效应空间内，以及不同β能够保持相同拟合。

**是否只是普通的固定效应基准组问题？**普通基准组省略后，我们的FE矩阵已经满列秩；额外加入D仍不增加秩。这条新增线性依赖涉及目标参数，不能靠重新命名基准组解决。

**为什么不删除年月固定效应，或改用趋势项？**替代规格可能恢复处理项的剩余变化，但也改变控制的时间变化和因果假设。它回答另一种设定下的问题，不否定原规格中的线性依赖。仓库 `code/03_robustness.py` 补做规格敏感性检查；下方同时展示其结果，不把改变设定后的系数称为原H2已修复。

**为什么不能解出州项和年月项，间接计算改革效应？**解出的是D的组成部分a和b，说明它能被控制项吸收；没有补出缺失组合中的Y。模型可以重新分配β与固定效应，不能据此唯一确定β。

**H3为正，是否要求短期连任效应也为正？**长期组的变化大于短期组，不决定短期组相对无连任组的变化方向。直接补做这一比较有解释价值，因果结论仍受其自身样本和趋势条件约束。

**短期组样本少，是否就必须报告或可以完全忽略？**样本问题应透明说明。补充比较有助于限定结论的适用范围，但我们不能仅由正文未报告它，推断作者动机或认定其违反某项报告义务。

**H2结果影响的是整篇论文吗？**直接受影响的是Figure 6末年处理效应及其对“临近选举效应更大”的支持。H2是一项动态预测；这一审计不能单独推翻全部理论链条，也不能证明总体连任效应不存在。

In [ ]:
from audit_lib import alternative_specifications
sensitivity = pd.DataFrame(alternative_specifications(h2))
sensitivity['beta_pp'] = sensitivity['beta'] * 100
sensitivity['se_pp'] = sensitivity['se'] * 100
display(sensitivity[['specification','identified','rank_controls','rank_controls_D','beta_pp','se_pp','p']])
assert not sensitivity.iloc[0]['identified']
assert not sensitivity.iloc[2]['identified']

## 7. 回填阅读报告与审计呈现

### 7.1 建议采用的证据顺序

1. 用一段说明原文研究问题和H1/H2/H3，并给出表号与印刷页码。
2. 链接本notebook，说明原始文件、版本、Python设定和复现边界。
3. 展示H1/H3可复现，再进入H2样本覆盖与独立处理变化。
4. 展示D恒等式、秩不变和相同拟合；将4.60/5.58放在额外参数约束的解释中。
5. 用补充记录说明缺失的信息类型，明确改变筛选不等于直接修复H2。
6. 短期组比较单列，点估计、少聚类推断和处理前检查同时呈现。
7. 以影响范围收束：哪些原文结论仍有点估计支持，哪些证据需要重新解释。

### 7.2 对现有阅读报告的措辞修订

| 原有或容易出现的表述 | 本审计采用的表述 |
|---|---|
| 作者没有说明省略短期比较的动机 | 正文未报告这组直接比较；我们补做，不推断动机 |
| p=0.117说明存在处理前差异 | 逐项结果有负向偏离；联合检验未在常用水平下拒绝共同为零 |
| 图5证明/确认平行趋势 | 未发现持续正向提前效应，不等于证明整体平行趋势 |
| 固定效应产生和改革项相同的系数 | 固定效应的线性组合可以精确表示处理变量的取值 |
| 上三行是反对角矩阵 | 上三行具有反对称关系 |
| 2016年6月是最后加入的一项 | 顺序重构中它成为冗余列，不是全部月份的最后一项 |
| 省略早期E窗口中的一个月 | 正常基准之外，额外省略中间日历窗口的月份项 |
| 处理变量的双重差分理应等于1 | 对图示标准两组、两期处理状态等于1；不泛指所有DID |
| 因为原始数据总量不够 | 原脚本筛选后的子样本缺少关键比较组合 |
| H2有问题，所以整篇论文结论错误 | H2末年估计在所述规格下不能独立识别；分别评价其他结果 |

说明中区分原文论断、复现数值与审计判断，避免将未核实的运行过程或动机解释写成既定事实。

### 7.3 尚待核实与公开复核入口

- 本次未取得原作者Stata日志，未执行原生Stata；关于实际省略列的过程保留为重构。
- 关于筛选选择的理由及作者回应，本notebook未纳入新的作者说明，不从公开文件中猜测动机。
- CTM主题数选择、重新训练和主题估计不确定性不属于本次已完成的验证。
- 仓库提供获取脚本、哈希、共享估计实现、聚合输出和本 notebook；原始数据仍只从 Dataverse 下载。

### 7.4 本次运行的关键结果

下面全部来自本次运行的变量，顺序与主审计和补充分析一致。

In [ ]:
final_checks = pd.DataFrame([
    {'检查':'完整原始数据','本次结果':f'{raw.shape[0]} 行 × {raw.shape[1]} 列'},
    {'检查':'H1 点估计（百分点）','本次结果':f'{h1["b"]*100:.6f}'},
    {'检查':'H3 点估计（百分点）','本次结果':f'{h3["b"]*100:.6f}'},
    {'检查':'H2末年恒等式不成立行数','本次结果':int(identity_error.ne(0).sum())},
    {'检查':'H2末年 FE / FE+D 秩','本次结果':f'{h2_records[2]["rank_FE"]} / {h2_records[2]["rank_FE_D"]}'},
    {'检查':'额外省略2016年6月 β（百分点）','本次结果':f'{constrained_models["额外省略2016年6月"].params.D*100:.6f}'},
    {'检查':'额外省略2014年7月 β（百分点）','本次结果':f'{constrained_models["额外省略2014年7月"].params.D*100:.6f}'},
    {'检查':'短期 vs 无连任（百分点）','本次结果':f'{short_results[0]["b"]*100:.6f}'},
    {'检查':'短期比较 wild bootstrap p','本次结果':f'{short_results[0]["wild_p"]:.6f}'},
    {'检查':'短期33切点联合 bootstrap p','本次结果':f'{short_joint_p:.6f}'}
])
display(final_checks)
print('全部关键数值断言通过。运行日期：', pd.Timestamp.now().isoformat(timespec='seconds'))